# Исследование зависимостей особенных характеристик песен на базе данных от 01.01.2015 до июля 2023 (включительно)

В данном исследовании будут рассмотрены следующие аспекты:
1. Стриминг (продвижение в плейлистах + общие стримы)
2. Победы и номинации на премии The Grammys
3. Успехи в чарте Billboard Hot 100 (число недель в "горячей сотне" и пиковые позиции)
4. Характеристики песен (energy, tempo, dancebility и т.п.)

Исследование базируется на датасетах, найденных на просторах портала Kaggle. Стоит отметить, что одного датасета, который покрыл бы потребности работы, не был найден и решено собрать свой из нескольких

Установим библиотку ftfy, необходимую для работы с "битыми" символами, которые важно обработать для гарантии корректного "мёрджа" данных из разных таблиц по полям "Artist" и "Track"

In [29]:
!pip install ftfy
!pip install rapidfuzz

Подключаемся к Google-диску для получения датасетов (удобно, чтобы не загружать каждый раз для одной сессии)

In [30]:
from google.colab import drive
drive.mount('/content/drive/')

path = "/content/drive/My Drive/datasets/"

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [31]:
import pandas as pd
import numpy as np
from IPython.display import display

df_23 = pd.read_excel(path + "spotify_2023.xlsx")
df_billboard = pd.read_csv(path + "hot100.csv", sep=',', encoding='latin1', engine='python')
df_grammy = pd.read_csv(path + "Grammy Award Nominees and Winners 1958-2024.csv", sep=',', encoding='latin1', engine='python')
df_digital_songs = pd.read_csv(path + "digital_songs.csv", sep=',', encoding='latin1', engine='python')
df_radio = pd.read_csv(path + "radio.csv", sep=',', encoding='latin1', engine='python')
df_streaming = pd.read_csv(path + "streaming_songs.csv", sep=',', encoding='latin1', engine='python')

Приведем ключевые столбцы к единому названию

In [32]:
df_23 = df_23.rename(columns={
    'track_name': 'Song',
    'artist(s)_name': 'Artist',
    'streams': 'Spotify Streams'
})

df_grammy = df_grammy.rename(columns={
    'Work': 'Song',
    'Nominee': 'Artist'
})

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

Удостоверимся, что числовые данные имеют нужный тип данных и уберем пропуски (уберем все строки с пустыми значениями для важных полей и вновсим среднее для неважных)

In [33]:
num_cols = ['released_year', 'released_month','released_day', 'in_spotify_playlists', 'in_spotify_charts',
       'Spotify Streams', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%',
       'instrumentalness_%', 'liveness_%', 'speechiness_%']
num_cols_mid = ['bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%',
       'instrumentalness_%', 'liveness_%', 'speechiness_%']
num_cols_zero = ['in_spotify_playlists', 'in_spotify_charts',
       'Spotify Streams', 'in_apple_playlists', 'in_apple_charts',
       'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts']
num_col_delete = ['released_year', 'released_month','released_day']
charts = [df_digital_songs, df_streaming, df_billboard, df_radio]
charts_data = ['Rank', 'Last Week', 'Peak Position']
cat_cols = ['key', 'mode']

#df_23
for col in num_cols:
    df_23[col] = (df_23[col].astype(str).str.replace(',', '.').str.replace('-', ''))
    df_23[col] = pd.to_numeric(df_23[col], errors='coerce')
    if col in num_cols_mid:
        df_23[col] = df_23[col].fillna(df_23[col].mean())
    elif col in num_cols_zero:
        df_23[col] = df_23[col].fillna(0)
    else:
        df_23[col] = df_23[col].dropna()
    df_23[col] = df_23[col].astype(int)


for col in cat_cols:
    df_23[col] = df_23[col].fillna(df_23[col].mode()[0])

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

#all df
for df in datasets:
    df = df.dropna(subset=['Song', 'Artist'])

#df_grammy
df_grammy = df_grammy.dropna(subset=['Winner', 'Award Name'])

#df_charts
def clean_chart_numeric(df):
    numeric_cols = ['Rank', 'Last Week', 'Peak Position', 'Weeks in Charts']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

for df in charts:
    df['Weeks in Charts'] = df['Weeks in Charts'].replace(['NaN', '', ' ', '-'], np.nan)
    df['Weeks in Charts'] = df['Weeks in Charts'].fillna(1)
    df['Weeks in Charts'] = df['Weeks in Charts'].astype(int)
    df = df.dropna(subset=charts_data)

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

for name, df_chart in [
    ("df_billboard", df_billboard),
    ("df_digital_songs", df_digital_songs),
    ("df_radio", df_radio),
    ("df_streaming", df_streaming)
]:
    if not df_chart.empty:
        globals()[name] = clean_chart_numeric(df_chart.copy())

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

Уберем из чартов все записи с песнями, кроме последних, поскольку последняя фиксированная запись по песне в чарте содержит в себе актуальную информацию о пике в топ-100 и числе недель в чарте. За одно удалим возможные дубликаты в df_23

In [34]:
df_23 = df_23.drop_duplicates(subset=['Song', 'Artist'], keep="last")
# df_billboard = df_billboard.drop_duplicates(subset=['Song', 'Artist'], keep="last")
# df_grammy = df_grammy.drop_duplicates(subset=['Song', 'Artist', 'Award Name'], keep="last")
# df_digital_songs = df_digital_songs.drop_duplicates(subset=['Song', 'Artist'], keep="last")
# df_radio = df_radio.drop_duplicates(subset=['Song', 'Artist'], keep="last")
# df_streaming = df_streaming.drop_duplicates(subset=['Song', 'Artist'], keep="last")

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

В датасетах чартов и Грэмми представлены данные за всю историю. Остановимся только на тех, что датируется не ранее 01.01.2020

In [35]:
year_start = 2016

df_grammy = df_grammy[df_grammy['Year'] >= year_start].copy()

for name, df in [
    ("df_billboard", df_billboard),
    ("df_digital_songs", df_digital_songs),
    ("df_radio", df_radio),
    ("df_streaming", df_streaming)
]:
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['year'] = df['Date'].dt.year
        df_filtered = df[df['year'] >= year_start].copy()
        globals()[name] = df_filtered
        print(f"{name}: оставлено {len(df_filtered):,} строк (с {year_start} года)")

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

df_billboard: оставлено 51,200 строк (с 2016 года)
df_digital_songs: оставлено 22,975 строк (с 2016 года)
df_radio: оставлено 25,600 строк (с 2016 года)
df_streaming: оставлено 25,600 строк (с 2016 года)


Обработаем столбец Artist в df_grammy, поскольку есть кейсы, когда указываются имена реальных людей, а не пвсевдоним артиста


> **Пример:** "billie eilish o'connell, finneas o'connell, songwriters (billie eilish)" - для мэтча далее нужно именно сценическое имя артиста, т.е. Billie Eilish в нижнем регистре



In [36]:
import re

def extract_artist_name(text):
    text = str(text)
    match = re.search(r"\((.*?)\)", text)
    return match.group(1) if match else text

df_grammy['Artist'] = df_grammy['Artist'].apply(extract_artist_name)

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

Обработка записей с несколькими исполнителями и названий, которые указаны по-разному

> **Пример**: в Spotify-2023 песня Билли Айлиш "What Was I Made For?" указана с припиской "[From The Motion Picture “Barbie”]" которая отсутствует в базе данных с историей чартов Биллборд



In [37]:
import re
import ftfy
from rapidfuzz import process, fuzz

def clean_text(s):
    return str(s).lower().strip()

import re

def normalize_artists(s):
    if not isinstance(s, str):
        s = str(s)

    s = s.lower()
    s = re.sub(r'\bfeaturing\b', ',', s)
    s = re.sub(r'\bfeat\.?\b', ',', s)
    s = re.sub(r'\b ft\.?\b', ',', s)
    s = re.sub(r'\bwith\b', ',', s)
    s = re.sub(r'[&|/;+]', ',', s)
    s = re.sub(r'[^a-z0-9,\s]', '', s)
    artists = [a.strip() for a in s.split(',') if a.strip()]
    blacklist = {'various artists', 'unknown artist'}
    artists = [a for a in artists if a not in blacklist]
    return sorted(set(artists))

def normalize_song(s):
    if not isinstance(s, str):
        s = str(s)

    s = s.lower().strip()
    s = re.sub(r'\[.*?\]', '', s)
    s = re.sub(r'\(\s*featuring.*?\)', '', s)
    s = re.sub(r'\(\s*feat\.?.*?\)', '', s)
    s = re.sub(r'\(\s*ft\.?.*?\)', '', s)
    s = re.sub(r'\(\s*remix.*?\)', '', s)
    s = re.sub(r'\(\s*explicit.*?\)', '', s)
    s = re.sub(r'\(\s*radio.*?\)', '', s)
    s = re.sub(r'\(\s*edit.*?\)', '', s)
    s = re.sub(r'\(\s*clean.*?\)', '', s)
    s = re.sub(r'\(\s*from.*?\)', '', s)
    s = re.sub(r'\s*-\s*(remix|radio edit|explicit|clean).*', '', s)
    s = re.sub(r'\bfeaturing\b.*$', '', s)
    s = re.sub(r'\bfeat\.?\b.*$', '', s)
    s = re.sub(r'\bft\.?\b.*$', '', s)
    s = re.sub(r'\bwith\b.*$', '', s)
    s = re.sub(r'[^\w\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()

    return s

    for pattern in junk_patterns:
        s = re.sub(pattern, '', s, flags=re.IGNORECASE)

    s = re.sub(r'[^\w\s]', '', s)

    s = re.sub(r'\s+', ' ', s).strip()
    return s


def clean_artist_list(artist_list):
    if not isinstance(artist_list, list):
        return []

    cleaned = []
    for a in artist_list:
        a = str(a).lower().strip()
        a = re.sub(r'\s*(featuring|feat\.?|ft\.?|&|with)\s*', ',', a, flags=re.IGNORECASE)
        a = re.sub(r'[^a-zа-я0-9\s,]', '', a)
        a = re.sub(r'\s+', ' ', a).strip()
        if a:
            cleaned.append(a)

    return sorted(set(cleaned))

def restore_artist_from_list(df):
    if 'artist_list' not in df.columns:
        return df
    df = df.copy()
    df['Artist'] = df['artist_list'].apply(
        lambda lst: ", ".join(clean_artist_list(lst)) if lst else ""
    )
    return df



datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

for df in datasets:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).apply(ftfy.fix_text).apply(clean_text)

    if 'Artist' in df.columns:
        df['artist_list'] = df['Artist'].apply(normalize_artists)

    if 'Song' in df.columns:
        df['Song'] = df['Song'].apply(normalize_song)


datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

for df in [df_billboard, df_digital_songs, df_radio, df_streaming, df_grammy]:
    if 'artist_list' in df.columns:
        df['Artist'] = df['artist_list'].apply(
            lambda x: ', '.join(x) if isinstance(x, list) else str(x)
        )

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

for df_name in ['df_23', 'df_billboard', 'df_digital_songs', 'df_radio', 'df_streaming', 'df_grammy']:
    if df_name in globals():
        globals()[df_name] = restore_artist_from_list(globals()[df_name])

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

for df in datasets:
  df['Song'] = df['Song'].str.replace("taylors version", "(taylor's version)")


datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

Начнем работу только с теми записями из df_grammy и df чартов. Для этого оставим только те песни, которые есть в df_23

In [38]:
valid_tracks = set(df_23['Song'])
valid_artists = set()
for artist_list in df_23['artist_list']:
    valid_artists.update(artist_list)

Уникальных треков в df_23: 782
Уникальных артистов в df_23: 587


Выделим ТОЛЬКО песенные номинации (убираем альбомы и профессиональные (артист, продюсер, сонграйтер) номинации)

In [39]:
song_keywords = ['song', 'record', 'performance', 'Song', 'Record', 'Performance']
df_grammy = df_grammy[df_grammy['Award Name'].str.contains('|'.join(song_keywords), case=False, na=False)].copy()

df_grammy = df_grammy[df_grammy['Song'].isin(valid_tracks)].copy()

important_categories = ['song of the year', 'record of the year', 'Song of the Year', 'Record of the Year']
df_grammy_major = df_grammy[df_grammy['Award Name'].str.contains('|'.join(important_categories), case=False, na=False)].copy()
df_grammy_major.to_csv("soty_roty.csv", sep=";", index=False, encoding='utf-8-sig')

def keep_best_grammy(group):
    wins = group[group['Winner'] == True]
    if not wins.empty:
        return wins.iloc[0]
    return group.iloc[0]

df_grammy = df_grammy.groupby(['Song', 'Year'], as_index=False).apply(keep_best_grammy)
df_grammy = df_grammy.reset_index(drop=True)

datasets = [df_23, df_billboard, df_grammy, df_digital_songs, df_radio, df_streaming]

/tmp/ipykernel_10936/4210642322.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_grammy = df_grammy.groupby(['Song', 'Year'], as_index=False).apply(keep_best_grammy)


Создадим "ключи", чтобы соотносить записи из разных df для дальнейшего мёрджа

In [40]:
import re
from rapidfuzz import fuzz, process

def create_key(row):
    artist = str(row.get('Artist', '')).lower().strip()
    song = str(row.get('Song', '')).lower().strip()
    return f"{artist} | {song}"

df_23['key'] = df_23.apply(create_key, axis=1)

for df_chart in [df_billboard, df_digital_songs, df_radio, df_streaming]:
    df_chart['key'] = df_chart.apply(create_key, axis=1)

df_23_keys = df_23['key'].tolist()

def aggregate_chart_last_n(df_chart, chart_name, n=5):
    if df_chart.empty:
        return pd.DataFrame(columns=['key', f'peak_{chart_name}', f'weeks_{chart_name}'])

    df = df_chart.copy()

    df_last = df.groupby('key', group_keys=False).apply(lambda x: x.tail(n))

    df_last.to_csv(chart_name + "123.csv", sep=";", index=False, encoding='utf-8-sig')

    agg = df_last.groupby('key').agg(
        **{
            f'peak_{chart_name}': ('Peak Position', 'min'),
            f'weeks_{chart_name}': ('Weeks in Charts', 'max')
        }
    ).reset_index()

    return agg

billboard_agg = aggregate_chart_last_n(df_billboard, 'hot100')
digital_agg   = aggregate_chart_last_n(df_digital_songs, 'digital')
radio_agg     = aggregate_chart_last_n(df_radio, 'radio')
streaming_agg = aggregate_chart_last_n(df_streaming, 'streaming')

df_final = df_23.copy()

for agg_df in [billboard_agg, digital_agg, radio_agg, streaming_agg]:
    if not agg_df.empty:
        df_final = df_final.merge(agg_df, on='key', how='left')

/tmp/ipykernel_10936/1411782642.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_last = df.groupby('key', group_keys=False).apply(lambda x: x.tail(n))
/tmp/ipykernel_10936/1411782642.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_last = df.groupby('key', group_keys=False).apply(lambda x: x.tail(n))
/tmp/ipykernel_10936/1411782642.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on 

In [41]:
if not df_grammy.empty:
    df_grammy['key'] = df_grammy.apply(create_key, axis=1)
    df_final = df_final.merge(df_grammy[['key', 'Winner']], on='key', how='left')
    df_final = df_final.rename(columns={'Winner': 'grammy_won'})
    df_final['grammy_nominated'] = df_final['grammy_won'].notna()

,Song,Artist,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,Spotify Streams,in_apple_playlists,...,peak_hot100,weeks_hot100,peak_digital,weeks_digital,peak_radio,weeks_radio,peak_streaming,weeks_streaming,grammy_won,grammy_nominated
0,seven,"jung kook, latto",2,2023,7,14,553,147,141381703,43,...,1.0,15.0,2.0,8.0,43.0,7.0,4.0,8.0,NaN,False
1,lala,myke towers,1,2023,3,23,1474,48,133716286,48,...,43.0,20.0,NaN,NaN,NaN,NaN,1.0,5.0,NaN,False
2,vampire,olivia rodrigo,1,2023,6,30,1397,113,140003974,94,...,1.0,35.0,2.0,14.0,6.0,35.0,1.0,32.0,False,True
3,cruel summer,"taylor swi,",1,2019,8,23,7858,100,800840817,116,...,1.0,54.0,1.0,31.0,1.0,48.0,1.0,61.0,NaN,False
4,where she goes,bad bunny,1,2023,5,18,3133,50,303236322,84,...,8.0,18.0,1.0,1.0,NaN,NaN,3.0,10.0,NaN,False


Приводим итоговые поля "в порядок"

In [42]:
df_final = df_final.fillna({
    'peak_hot100': 999, 'weeks_hot100': 0,
    'peak_digital': 999, 'weeks_digital': 0,
    'peak_radio': 999, 'weeks_radio': 0,
    'peak_streaming': 999, 'weeks_streaming': 0,
    'grammy_nominated': False, 'grammy_won': False
})

int_cols = ['peak_hot100', 'peak_digital', 'peak_radio', 'peak_streaming',
            'weeks_hot100', 'weeks_digital', 'weeks_radio', 'weeks_streaming']
for col in int_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(999 if 'peak' in col else 0).astype(int)

bool_cols = ['grammy_nominated', 'grammy_won']
for col in bool_cols:
    df_final[col] = df_final[col].fillna(False).astype(bool)

for col in int_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(999 if 'peak' in col else 0).astype(int)


/tmp/ipykernel_10936/2458708655.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df_final.fillna({


Сохраняем итоговый датасет в формате CSV

In [43]:
# df_billboard.to_csv("hot100_short.csv", sep=";", index=False, encoding='utf-8-sig')
# df_digital_songs.to_csv("digital_short.csv", sep=";", index=False, encoding='utf-8-sig')
# df_radio.to_csv("radio_short.csv", sep=";", index=False, encoding='utf-8-sig')
# df_streaming.to_csv("streaming_short.csv", sep=";", index=False, encoding='utf-8-sig')
# df_grammy.to_csv("grammy_short.csv", sep=";", index=False, encoding='utf-8-sig')
# df_23.to_csv("spotify_short.csv", sep=";", index=False, encoding='utf-8-sig')

In [44]:
df_final.to_csv("final_2023_songs.csv", sep=";", index=False, encoding='utf-8-sig')

Работаем с итоговыми данными, вычисляем

In [45]:
winners = df_final[df_final['grammy_won'] == True]
display(winners[['Song']])

,Song
12,flowers
77,unholy
95,all my life
97,snooze
127,watermelon sugar
178,shape of you
230,cuff it
245,easy on me
469,family ties
474,shallow


In [46]:
metrics = ['bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%']

for col in metrics:
  mid = (df_final[col].mean()).round()
  print("mid " + col + " " + str(int(mid)))

mid bpm 122
mid danceability_% 68
mid valence_% 51
mid energy_% 65
mid acousticness_% 26
mid instrumentalness_% 2
mid liveness_% 18
mid speechiness_% 10


In [47]:
df_final['Spotify Streams'] = df_final['Spotify Streams'].astype(int)

top20 = df_final.sort_values(by='Spotify Streams', ascending=False).head(20)

metrics = ['bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%']

for col in metrics:
  mid = (top20[col].mean()).round()
  print("mid " + col + " " + str(int(mid)))

mid bpm 115
mid danceability_% 64
mid valence_% 52
mid energy_% 59
mid acousticness_% 34
mid instrumentalness_% 0
mid liveness_% 16
mid speechiness_% 7


In [48]:
amount = 20

spotify = df_final.sort_values(by='in_spotify_playlists', ascending=False).head(20)
deezer = df_final.sort_values(by='in_deezer_playlists', ascending=False).head(20)
apple = df_final.sort_values(by='in_apple_playlists', ascending=False).head(20)

In [49]:
print(spotify['released_year'].mean())
print(deezer['released_year'].mean())
print(apple['released_year'].mean())


2003.15
2001.05
2015.25


In [50]:
import ast

def safe_eval(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

df = df_final.copy()
df['artist_list'] = df['artist_list'].apply(safe_eval)
df_artists = df.explode('artist_list')
df_artists = df_artists.rename(columns={'artist_list': 'artist_single'})

cols = [
    'in_spotify_playlists',
    'in_apple_playlists',
    'in_deezer_playlists'
]

for col in cols:
    df_artists[col] = pd.to_numeric(df_artists[col], errors='coerce').fillna(0)

df_result = df_artists.groupby('artist_single')[cols].sum().reset_index()

df_result['all_playlists'] = df_result['in_spotify_playlists'] + df_result['in_apple_playlists'] + df_result['in_deezer_playlists']

df_result.to_csv("artists_playlists.csv", sep=";", index=False, encoding='utf-8-sig')

In [51]:
df_hits = df_final[df_final['peak_hot100'] == 1]
df_final.shape

(793, 35)

In [52]:
top200 = df_final.sort_values(by='Spotify Streams', ascending=False).head(200)
top200.to_csv("top200byStreams.csv", sep=";", index=False, encoding='utf-8-sig')

In [53]:
top10 = top200.head(10)
top10[['Song', 'Spotify Streams', 'Artist', 'peak_hot100', 'peak_radio', 'peak_digital', 'peak_streaming']]

,Song,Spotify Streams,Artist,peak_hot100,peak_radio,peak_digital,peak_streaming
55,blinding lights,3703895074,the weeknd,1,1,1,1
178,shape of you,3562543890,ed sheeran,1,1,1,1
86,someone you loved,2887241814,lewis capaldi,1,1,2,7
460,dance monkey,2864791672,tones and i,4,11,2,4
41,sunflower spiderman into the spiderverse,2808096550,"post malone, swae lee",1,7,1,1
162,one dance,2713922350,"drake, kyla, wizkid",1,1,1,1
84,stay,2665343922,"justin bieber, the kid laroi",1,1,3,1
140,believer,2594040133,imagine dragons,4,4,2,13
565,closer,2591224264,"halsey, the chainsmokers",1,1,1,1
48,starboy,2565529693,"da,punk, the weeknd",1,2,1,2


In [54]:
metrics = ['bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%']

for col in metrics:
  mid = (top20[col].mean()).round()
  print("mid " + col + " " + str(int(mid)))

mid bpm 115
mid danceability_% 64
mid valence_% 52
mid energy_% 59
mid acousticness_% 34
mid instrumentalness_% 0
mid liveness_% 16
mid speechiness_% 7


In [61]:
roty = df_grammy_major[df_grammy_major["Award Name"] == "record of the year"]
print(f"Всего номинантов: {roty.shape[0]}")
roty_winners = roty[roty["Winner"] == True]
print(f"Победителей: {roty_winners.shape[0]}")

soty = df_grammy_major[df_grammy_major["Award Name"] == "song of the year"]
print(f"Всего номинантов: {soty.shape[0]}")
soty_winners = soty[soty["Winner"] == True]
print(f"Победителей: {soty_winners.shape[0]}")



Всего номинантов: 16
Победителей: 3
Всего номинантов: 19
Победителей: 1


,Unnamed: 0,Year,Ceremony,Award ID,Award Type,Award Name,Song,Artist,Winner,artist_list
23511,23511,2021,64,629,work,song of the year,leave the door open,silk sonic,True,[silk sonic]
